# Analiz chekov pekarni Tukaya 62A

Dannye: 58,484 zapisej, 89 dnej (02.01.2026 - 31.03.2026), 221 tovar, 1 pekarniya

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

# Load
df = pd.read_csv('../data/raw/checks_g_t_62a.csv', sep=';', encoding='cp1251')
for col in ['Цена', 'Кол-во', 'Сумма']:
    df[col] = df[col].str.replace(' ', '').str.replace(',', '.').astype(float)

df['Дата продажи'] = pd.to_datetime(df['Дата продажи'], format='%d.%m.%Y')
df['Дата время чека'] = pd.to_datetime(df['Дата время чека'], format='%d.%m.%Y %H:%M:%S')

# Tolko prodazhi (bez vozvratov i otmen)
print(f"Vid sobytiya: {df['Вид события по кассе'].value_counts().to_dict()}")
df = df[df['Вид события по кассе'] == 'Продажа'].copy()
print(f"Posle filtratsii: {len(df):,} zapisej")

# Derived columns
df['hour'] = df['Дата время чека'].dt.hour
df['dow'] = df['Дата время чека'].dt.dayofweek
df['dow_name'] = df['dow'].map({0:'Pn',1:'Vt',2:'Sr',3:'Cht',4:'Pt',5:'Sb',6:'Vs'})
df['week'] = df['Дата время чека'].dt.isocalendar().week.astype(int)
df['month'] = df['Дата время чека'].dt.month

df.shape

## 1. Dnevnaya dinamika: vyruchka, qty, cheki

In [ ]:
daily = df.groupby('Дата продажи').agg(
    qty=('Кол-во', 'sum'),
    revenue=('Сумма', 'sum'),
    checks=('Дата время чека', 'nunique'),
    items=('Номенклатура', 'count')
).sort_index()

daily['avg_check'] = daily['revenue'] / daily['checks']
daily['dow'] = daily.index.dayofweek

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Revenue
ax = axes[0]
colors = ['#2196F3' if d < 5 else '#FF5722' for d in daily['dow']]
ax.bar(daily.index, daily['revenue'], color=colors, alpha=0.7)
ax.axhline(daily['revenue'].mean(), color='black', ls='--', lw=1, label=f"Mean={daily['revenue'].mean():,.0f}")
ax.set_ylabel('Revenue (rub)')
ax.set_title('Dnevnaya vyruchka (sinij=budni, krasnyj=vyhodnye)')
ax.legend()

# Qty
ax = axes[1]
ax.bar(daily.index, daily['qty'], color=colors, alpha=0.7)
ax.axhline(daily['qty'].mean(), color='black', ls='--', lw=1, label=f"Mean={daily['qty'].mean():,.0f}")
ax.set_ylabel('Qty (sht)')
ax.set_title('Dnevnoe kol-vo prodannyh tovarov')
ax.legend()

# Avg check
ax = axes[2]
ax.plot(daily.index, daily['avg_check'], 'o-', ms=3, color='#4CAF50')
ax.axhline(daily['avg_check'].mean(), color='black', ls='--', lw=1, label=f"Mean={daily['avg_check'].mean():,.0f}")
ax.set_ylabel('Avg check (rub)')
ax.set_title('Srednij chek')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

print(f"\nStatistika:")
print(f"  Revenue/den: mean={daily['revenue'].mean():,.0f}, std={daily['revenue'].std():,.0f}")
print(f"  Qty/den:     mean={daily['qty'].mean():,.0f}, std={daily['qty'].std():,.0f}")
print(f"  Chekov/den:  mean={daily['checks'].mean():,.0f}")
print(f"  Avg chek:    mean={daily['avg_check'].mean():,.0f} rub")

## 2. Nedelnaya sezonnost (po dnyam nedeli)

In [ ]:
dow_names = ['Pn', 'Vt', 'Sr', 'Cht', 'Pt', 'Sb', 'Vs']
dow_stats = daily.groupby('dow').agg(
    qty_mean=('qty', 'mean'),
    qty_std=('qty', 'std'),
    rev_mean=('revenue', 'mean'),
    rev_std=('revenue', 'std'),
    check_mean=('checks', 'mean'),
    avg_check=('avg_check', 'mean'),
    n_days=('qty', 'count')
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Revenue by DOW
ax = axes[0]
colors = ['#2196F3']*5 + ['#FF5722']*2
ax.bar(range(7), dow_stats['rev_mean'], yerr=dow_stats['rev_std'], 
       color=colors, alpha=0.7, capsize=5)
ax.set_xticks(range(7))
ax.set_xticklabels(dow_names)
ax.set_ylabel('Revenue (rub)')
ax.set_title('Srednaya vyruchka po dnyam nedeli')

# Qty by DOW
ax = axes[1]
ax.bar(range(7), dow_stats['qty_mean'], yerr=dow_stats['qty_std'],
       color=colors, alpha=0.7, capsize=5)
ax.set_xticks(range(7))
ax.set_xticklabels(dow_names)
ax.set_ylabel('Qty (sht)')
ax.set_title('Srednee kol-vo po dnyam nedeli')

# Avg check by DOW
ax = axes[2]
ax.bar(range(7), dow_stats['avg_check'], color=colors, alpha=0.7)
ax.set_xticks(range(7))
ax.set_xticklabels(dow_names)
ax.set_ylabel('Avg check (rub)')
ax.set_title('Srednij chek po dnyam nedeli')

plt.tight_layout()
plt.show()

# Tablitsa
overall_rev = dow_stats['rev_mean'].mean()
print(f"\n{'DOW':<5} {'Rev':>8} {'Idx':>6} {'Qty':>6} {'AvgChk':>7} {'Days':>4}")
for d in range(7):
    r = dow_stats.loc[d]
    idx = r['rev_mean'] / overall_rev * 100
    print(f"{dow_names[d]:<5} {r['rev_mean']:>8,.0f} {idx:>5.0f}% {r['qty_mean']:>6.0f} {r['avg_check']:>7.0f} {r['n_days']:>4.0f}")

## 3. Chasovoj pattern prodazh (dnevnoj profil)

In [ ]:
# Overall hourly pattern
hourly_all = df.groupby('hour').agg(
    qty=('Кол-во', 'sum'),
    revenue=('Сумма', 'sum'),
    checks=('Дата время чека', 'count')
)

# Hourly by DOW type (budni vs vyhodnye)
df['is_weekend'] = df['dow'] >= 5
hourly_wd = df[~df['is_weekend']].groupby('hour')['Кол-во'].sum()
hourly_we = df[df['is_weekend']].groupby('hour')['Кол-во'].sum()

n_workdays = daily[daily['dow'] < 5].shape[0]
n_weekends = daily[daily['dow'] >= 5].shape[0]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Absolute
ax = axes[0]
ax.bar(hourly_all.index, hourly_all['qty'], color='#2196F3', alpha=0.7)
ax.set_xlabel('Chas')
ax.set_ylabel('Total qty za 3 mes.')
ax.set_title('Obshij profil prodazh po chasam')
ax.set_xticks(range(5, 21))

# Budni vs Vyhodnye (normalized per day)
ax = axes[1]
hours = range(5, 21)
wd_norm = [hourly_wd.get(h, 0) / n_workdays for h in hours]
we_norm = [hourly_we.get(h, 0) / n_weekends for h in hours]

x = np.arange(len(hours))
w = 0.35
ax.bar(x - w/2, wd_norm, w, label=f'Budni (n={n_workdays})', color='#2196F3', alpha=0.7)
ax.bar(x + w/2, we_norm, w, label=f'Vyhodnye (n={n_weekends})', color='#FF5722', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels([f'{h}:00' for h in hours], rotation=45)
ax.set_ylabel('Qty / den')
ax.set_title('Profil prodazh: budni vs vyhodnye (na 1 den)')
ax.legend()

plt.tight_layout()
plt.show()

# Peak hours
total = hourly_all['qty'].sum()
print(f"\nPikovye chasy (>8% ot total):")
for h in hourly_all.index:
    pct = hourly_all.loc[h, 'qty'] / total * 100
    if pct > 8:
        print(f"  {h:02d}:00 - {pct:.1f}%")

## 4. Chasovoj profil po kategoriyam tovarov (TOP-10)

In [ ]:
# TOP-10 products by qty
top10 = df.groupby('Номенклатура')['Кол-во'].sum().nlargest(10).index.tolist()

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.ravel()

for i, prod in enumerate(top10):
    ax = axes[i]
    prod_df = df[df['Номенклатура'] == prod]
    hourly_prod = prod_df.groupby('hour')['Кол-во'].sum()
    
    ax.bar(hourly_prod.index, hourly_prod.values, color='#4CAF50', alpha=0.7)
    ax.set_title(prod[:20], fontsize=9)
    ax.set_xlim(5.5, 20.5)
    ax.set_xticks([6, 9, 12, 15, 18])
    if i >= 5:
        ax.set_xlabel('Chas')
    if i % 5 == 0:
        ax.set_ylabel('Qty')

plt.suptitle('Chasovoj profil prodazh: TOP-10 tovarov', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 5. Kumulyativnyj profil prodazh (% ot dnevnogo itoga po chasam)

In [ ]:
# Cumulative intra-day sales profile
# For each day, compute hourly cumulative share
daily_hour = df.groupby(['Дата продажи', 'hour'])['Кол-во'].sum().reset_index()
daily_total = df.groupby('Дата продажи')['Кол-во'].sum().rename('day_total')
daily_hour = daily_hour.merge(daily_total, on='Дата продажи')
daily_hour['share'] = daily_hour['Кол-во'] / daily_hour['day_total']

# Average cumulative by hour
avg_share = daily_hour.groupby('hour')['share'].mean()
cum_share = avg_share.cumsum()

# Same but budni/vyhodnye
daily_hour['dow'] = pd.to_datetime(daily_hour['Дата продажи']).dt.dayofweek
daily_hour['is_we'] = daily_hour['dow'] >= 5

cum_wd = daily_hour[~daily_hour['is_we']].groupby('hour')['share'].mean().cumsum()
cum_we = daily_hour[daily_hour['is_we']].groupby('hour')['share'].mean().cumsum()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cum_share.index, cum_share.values * 100, 'ko-', ms=6, lw=2, label='Vse dni')
ax.plot(cum_wd.index, cum_wd.values * 100, 's--', color='#2196F3', ms=5, label='Budni')
ax.plot(cum_we.index, cum_we.values * 100, '^--', color='#FF5722', ms=5, label='Vyhodnye')

ax.axhline(50, color='gray', ls=':', alpha=0.5)
ax.axhline(80, color='gray', ls=':', alpha=0.5)
ax.set_xlabel('Chas')
ax.set_ylabel('Kumulyativnaya dolya prodazh (%)')
ax.set_title('Kumulyativnyj profil: kakaya dolya dnevnyh prodazh sdelana k kazhdomu chasu')
ax.set_xticks(range(5, 21))
ax.set_xticklabels([f'{h}:00' for h in range(5, 21)], rotation=45)
ax.legend()
ax.grid(True, alpha=0.3)

# Annotate 50% and 80% hours
for pct in [50, 80]:
    h_all = cum_share[cum_share * 100 >= pct].index[0]
    ax.annotate(f'{pct}% k {h_all}:00', xy=(h_all, pct), fontsize=9,
                xytext=(h_all + 1, pct + 3), arrowprops=dict(arrowstyle='->', color='gray'))

plt.tight_layout()
plt.show()

## 6. Svezhest: vchera vs segodnya

In [ ]:
# Svezhest analysis
fresh_df = df[df['Свежесть'].isin(['Свежий', 'Вчерашний'])].copy()

# By hour
fresh_hour = fresh_df.groupby(['hour', 'Свежесть'])['Кол-во'].sum().unstack(fill_value=0)
fresh_hour_pct = fresh_hour.div(fresh_hour.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
fresh_hour.plot(kind='bar', stacked=True, ax=ax, color=['#4CAF50', '#FFC107'], alpha=0.8)
ax.set_xlabel('Chas')
ax.set_ylabel('Qty')
ax.set_title('Prodazhi po svezhesti i chasam (absolyutnye)')
ax.legend(title='Svezhest')

ax = axes[1]
if 'Вчерашний' in fresh_hour_pct.columns:
    ax.bar(fresh_hour_pct.index, fresh_hour_pct['Вчерашний'], color='#FFC107', alpha=0.8)
    ax.set_xlabel('Chas')
    ax.set_ylabel('% Vchera')
    ax.set_title('Dolya "Vcherashnij" po chasam')
    ax.set_xticks(range(5, 21))

plt.tight_layout()
plt.show()

# Overall stats
fresh_total = fresh_df.groupby('Свежесть')['Кол-во'].sum()
print(f"\nObshaya dolya:")
for s, q in fresh_total.items():
    print(f"  {s}: {q:.0f} sht ({100*q/fresh_total.sum():.1f}%)")

# Top products sold as "Vcherashnij"
vcher = df[df['Свежесть'] == 'Вчерашний']
vcher_top = vcher.groupby('Номенклатура')['Кол-во'].sum().nlargest(10)
print(f"\nTOP-10 tovarov 'Vcherashnij':")
for name, q in vcher_top.items():
    total_prod = df[df['Номенклатура'] == name]['Кол-во'].sum()
    print(f"  {name:<35} {q:>6.0f} sht ({100*q/total_prod:.1f}% ot vsekh prodazh)")

## 7. Nedelnaya dinamika (po nedelyam)

In [ ]:
# Weekly trends
df['year_week'] = df['Дата время чека'].dt.strftime('%Y-W%W')
weekly = df.groupby('year_week').agg(
    qty=('Кол-во', 'sum'),
    revenue=('Сумма', 'sum'),
    checks=('Дата время чека', 'nunique'),
    unique_items=('Номенклатура', 'nunique'),
    days=('Дата продажи', 'nunique')
).sort_index()

# Normalize per day (some weeks have fewer days)
weekly['qty_per_day'] = weekly['qty'] / weekly['days']
weekly['rev_per_day'] = weekly['revenue'] / weekly['days']

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

ax = axes[0]
ax.plot(range(len(weekly)), weekly['rev_per_day'], 'o-', color='#2196F3', lw=2)
ax.set_ylabel('Revenue / day')
ax.set_title('Vyruchka na den po nedelyam')
ax.set_xticks(range(len(weekly)))
ax.set_xticklabels(weekly.index, rotation=45, ha='right', fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(range(len(weekly)), weekly['qty_per_day'], 'o-', color='#4CAF50', lw=2)
ax.set_ylabel('Qty / day')
ax.set_title('Kol-vo prodannyh tovarov na den po nedelyam')
ax.set_xticks(range(len(weekly)))
ax.set_xticklabels(weekly.index, rotation=45, ha='right', fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. TOP tovarov: dinamika po dnyam

In [ ]:
top5 = df.groupby('Номенклатура')['Кол-во'].sum().nlargest(5).index.tolist()

fig, axes = plt.subplots(5, 1, figsize=(14, 16), sharex=True)

for i, prod in enumerate(top5):
    ax = axes[i]
    prod_daily = df[df['Номенклатура'] == prod].groupby('Дата продажи')['Кол-во'].sum()
    
    # Fill missing days with 0
    all_days = pd.date_range(df['Дата продажи'].min(), df['Дата продажи'].max())
    prod_daily = prod_daily.reindex(all_days, fill_value=0)
    
    colors = ['#FF5722' if d.dayofweek >= 5 else '#2196F3' for d in prod_daily.index]
    ax.bar(prod_daily.index, prod_daily.values, color=colors, alpha=0.7)
    ax.axhline(prod_daily.mean(), color='black', ls='--', lw=1)
    ax.set_ylabel('Qty')
    ax.set_title(f'{prod}  (mean={prod_daily.mean():.1f}/den, total={prod_daily.sum():.0f})', fontsize=10)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))
axes[-1].xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
plt.xticks(rotation=45)
plt.suptitle('TOP-5 tovarov: dnevnye prodazhi', fontsize=13, y=1.0)
plt.tight_layout()
plt.show()

## 9. Heatmap: chas x den nedeli

In [ ]:
# Heatmap: hour x dow
pivot = df.groupby(['dow', 'hour'])['Кол-во'].sum().reset_index()

# Normalize by number of days per DOW
dow_counts = daily.groupby('dow').size()
pivot = pivot.merge(dow_counts.rename('n_days').reset_index(), on='dow')
pivot['qty_avg'] = pivot['Кол-во'] / pivot['n_days']

heatmap_data = pivot.pivot(index='dow', columns='hour', values='qty_avg').fillna(0)

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(heatmap_data.values, aspect='auto', cmap='YlOrRd')

ax.set_yticks(range(7))
ax.set_yticklabels(['Pn', 'Vt', 'Sr', 'Cht', 'Pt', 'Sb', 'Vs'])
ax.set_xticks(range(len(heatmap_data.columns)))
ax.set_xticklabels([f'{h}:00' for h in heatmap_data.columns])
ax.set_xlabel('Chas')
ax.set_title('Heatmap: srednie prodazhi (qty) po chasam i dnyam nedeli')

# Add numbers
for i in range(heatmap_data.shape[0]):
    for j in range(heatmap_data.shape[1]):
        val = heatmap_data.values[i, j]
        if val > 0:
            color = 'white' if val > heatmap_data.values.max() * 0.6 else 'black'
            ax.text(j, i, f'{val:.0f}', ha='center', va='center', fontsize=7, color=color)

plt.colorbar(im, label='Avg qty/day')
plt.tight_layout()
plt.show()

## 10. Analiz prodavtsov i smen

In [ ]:
# Sellers analysis
seller_stats = df.groupby('Продавец').agg(
    checks=('Дата время чека', 'count'),
    qty=('Кол-во', 'sum'),
    revenue=('Сумма', 'sum'),
    days=('Дата продажи', 'nunique'),
    avg_hour=('hour', 'mean'),
    min_hour=('hour', 'min'),
    max_hour=('hour', 'max')
)
seller_stats['rev_per_day'] = seller_stats['revenue'] / seller_stats['days']
seller_stats['qty_per_day'] = seller_stats['qty'] / seller_stats['days']

print("Prodavtsy:")
for name, r in seller_stats.iterrows():
    print(f"  {name}")
    print(f"    Dnej: {r['days']:.0f}, chekov: {r['checks']:.0f}, qty: {r['qty']:.0f}")
    print(f"    Revenue: {r['revenue']:,.0f} ({r['rev_per_day']:,.0f}/den)")
    print(f"    Chasy: {r['min_hour']:.0f}:00 - {r['max_hour']:.0f}:00 (srednij {r['avg_hour']:.1f})")

# Hourly pattern by seller
fig, ax = plt.subplots(figsize=(14, 5))
for name in df['Продавец'].unique():
    seller_hourly = df[df['Продавец'] == name].groupby('hour')['Кол-во'].sum()
    ax.plot(seller_hourly.index, seller_hourly.values, 'o-', label=name[:25], ms=5)

ax.set_xlabel('Chas')
ax.set_ylabel('Total qty')
ax.set_title('Profil prodazh po prodavtsam')
ax.legend()
ax.set_xticks(range(5, 21))
plt.tight_layout()
plt.show()

## 11. Srednij chek po chasam i vremeni

In [ ]:
# Group by check (unique datetime = one check)
check_level = df.groupby(['Дата время чека']).agg(
    items=('Номенклатура', 'count'),
    qty=('Кол-во', 'sum'),
    total=('Сумма', 'sum')
).reset_index()
check_level['hour'] = check_level['Дата время чека'].dt.hour
check_level['dow'] = check_level['Дата время чека'].dt.dayofweek

print(f"Vsego chekov: {len(check_level):,}")
print(f"Srednij chek: {check_level['total'].mean():.0f} rub")
print(f"Median chek: {check_level['total'].median():.0f} rub")
print(f"Srednee pozitsij v cheke: {check_level['items'].mean():.1f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Avg check by hour
ax = axes[0]
check_hourly = check_level.groupby('hour')['total'].mean()
ax.plot(check_hourly.index, check_hourly.values, 'o-', color='#2196F3', lw=2)
ax.set_xlabel('Chas')
ax.set_ylabel('Avg check (rub)')
ax.set_title('Srednij chek po chasam')
ax.set_xticks(range(5, 21))
ax.grid(True, alpha=0.3)

# Check distribution
ax = axes[1]
ax.hist(check_level['total'], bins=50, color='#4CAF50', alpha=0.7, edgecolor='white')
ax.axvline(check_level['total'].mean(), color='red', ls='--', label=f"Mean={check_level['total'].mean():.0f}")
ax.axvline(check_level['total'].median(), color='blue', ls='--', label=f"Median={check_level['total'].median():.0f}")
ax.set_xlabel('Check total (rub)')
ax.set_ylabel('Count')
ax.set_title('Raspredelenie summ chekov')
ax.legend()

plt.tight_layout()
plt.show()

## 12. Summary

In [ ]:
print("=" * 60)
print("SUMMARY: Pekarnya Tukaya 62A")
print("=" * 60)
print(f"Period: {df['Дата продажи'].min().date()} -- {df['Дата продажи'].max().date()} ({df['Дата продажи'].nunique()} dnej)")
print(f"Zapisej: {len(df):,} (tolko prodazhi)")
print(f"Tovarov: {df['Номенклатура'].nunique()}")
print(f"Prodavtsov: {df['Продавец'].nunique()}")
print(f"")
print(f"Revenue/den:  {daily['revenue'].mean():,.0f} rub")
print(f"Qty/den:      {daily['qty'].mean():,.0f} sht")
print(f"Chekov/den:   {daily['checks'].mean():,.0f}")
print(f"Avg chek:     {daily['avg_check'].mean():,.0f} rub")
print(f"")
print(f"Vchera: {fresh_df[fresh_df['Свежесть']=='Вчерашний']['Кол-во'].sum():.0f} sht "
      f"({100*fresh_df[fresh_df['Свежесть']=='Вчерашний']['Кол-во'].sum()/fresh_df['Кол-во'].sum():.1f}%)")
print(f"")
print("Nedelnaya sezonnost:")
overall = daily['qty'].mean()
for d in range(7):
    m = daily[daily['dow']==d]['qty'].mean()
    print(f"  {['Pn','Vt','Sr','Cht','Pt','Sb','Vs'][d]}: {m:.0f} ({m/overall*100:.0f}% ot srednego)")

print(f"\nPikovye chasy: 7:00, 8:00, 11:00-13:00")
print(f"Rabota: 6:00-19:00")

## 13. Censored demand: kogda tovar zakonchilsya?

Esli poslednaya prodazha tovara byla zadolgo do zakrytiya pekarni — tovar zakonchilsya i spros ne udovletvoren.
Eto pryamoj signal censored demand iz chekov, bez kostyley.

In [ ]:
# === CENSORED DEMAND: kogda tovar zakonchilsya? ===

# Dlya kazhdogo tovara v kazhdyj den: vremya poslednij prodazhi
# Pekarnya rabotaet primerno 6:00-19:00
# Esli poslednaya prodazha v 13:00 — tovar zakonchilsya za 6 chasov do zakrytiya

CLOSING_HOUR = 19  # primernoe vremya zakrytiya

# TOP-30 tovarov (iskluchaem paket i napitki)
exclude_keywords = ['Пакет', 'Капучино', 'Латте', 'Чай', 'Американо', 'Какао', 'Pulpy',
                    'Кола', 'Вода', 'Сок', 'Морс', 'Компот', 'Раф']
mask_food = ~df['Номенклатура'].str.contains('|'.join(exclude_keywords), case=False, na=False)
top30_food = df[mask_food].groupby('Номенклатура')['Кол-во'].sum().nlargest(30).index.tolist()

# Dlya kazhdogo tovara x den: poslednaya prodazha
last_sale = (df[df['Номенклатура'].isin(top30_food)]
             .groupby(['Дата продажи', 'Номенклатура'])['hour']
             .max()
             .reset_index()
             .rename(columns={'hour': 'last_hour'}))

last_sale['hours_before_close'] = CLOSING_HOUR - last_sale['last_hour']

# Schitaem "rannyj stop" = poslednaya prodazha do 15:00 (>4 chasa do zakrytiya)
last_sale['early_stop'] = last_sale['last_hour'] <= 14

# Statistika po tovaram
stop_stats = last_sale.groupby('Номенклатура').agg(
    total_days=('last_hour', 'count'),
    mean_last_hour=('last_hour', 'mean'),
    early_stop_days=('early_stop', 'sum'),
    mean_hours_before_close=('hours_before_close', 'mean')
)
stop_stats['early_stop_pct'] = 100 * stop_stats['early_stop_days'] / stop_stats['total_days']
stop_stats = stop_stats.sort_values('early_stop_pct', ascending=False)

print("TOP-30 eda: kak chasto tovar zakanchivaetsya rano (poslednaya prodazha <= 14:00)")
print(f"\n{'Tovar':<35} {'Dnej':>5} {'SrPosled':>9} {'RanStop':>8} {'%':>6}")
print("-" * 70)
for name, r in stop_stats.iterrows():
    marker = " <<<" if r['early_stop_pct'] > 20 else ""
    print(f"{name:<35} {r['total_days']:>5.0f} {r['mean_last_hour']:>8.1f}:00 "
          f"{r['early_stop_days']:>7.0f} {r['early_stop_pct']:>5.1f}%{marker}")

# Vizualizatsiya
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Raspredelenie "poslednij chas prodazhi" dlya TOP-10 po early_stop
ax = axes[0]
top_early = stop_stats.head(10).index.tolist()
for prod in top_early:
    prod_hours = last_sale[last_sale['Номенклатура'] == prod]['last_hour']
    ax.hist(prod_hours, bins=range(6, 21), alpha=0.4, label=prod[:18])
ax.set_xlabel('Poslednij chas prodazhi')
ax.set_ylabel('Kol-vo dnej')
ax.set_title('Raspredelenie poslednij prodazhi\n(TOP-10 po ranniemu okonchaniyu)')
ax.axvline(14, color='red', ls='--', lw=2, label='14:00 granitsa')
ax.legend(fontsize=7, loc='upper left')
ax.set_xticks(range(6, 20))

# 2. Srednij poslednij chas prodazhi po tovaram
ax = axes[1]
sorted_stats = stop_stats.sort_values('mean_last_hour')
colors = ['#FF5722' if v <= 14 else '#FFC107' if v <= 16 else '#4CAF50' 
          for v in sorted_stats['mean_last_hour']]
y_pos = range(len(sorted_stats))
ax.barh(y_pos, sorted_stats['mean_last_hour'], color=colors, alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels([n[:25] for n in sorted_stats.index], fontsize=7)
ax.set_xlabel('Srednij poslednij chas prodazhi')
ax.set_title('Srednij chas poslednij prodazhi po tovaram\n(krasnyj=zakanchivaetsya rano)')
ax.axvline(14, color='red', ls='--', lw=1)
ax.axvline(CLOSING_HOUR, color='black', ls=':', lw=1, label='Zakrytie')
ax.legend()

plt.tight_layout()
plt.show()

# Obshaya statistika
total_pairs = len(last_sale)
early_total = last_sale['early_stop'].sum()
print(f"\n--- ITOG ---")
print(f"Vsego tovar-dnej (TOP-30): {total_pairs:,}")
print(f"Iz nih rannij stop (<=14:00): {early_total} ({100*early_total/total_pairs:.1f}%)")
print(f"Eto znachit: v {100*early_total/total_pairs:.0f}% sluchaev tovar zakonchilsya "
      f"minimum za {CLOSING_HOUR-14} chasov do zakrytiya")

## 14. Profil prodazh po chasam dlya kazhdogo tovara (baza dlya Production Plan)

Kumulyativnyj profil dlya TOP tovarov: k kakomu chasu prodaetsya kakaya dolya.
Eto baza dlya Production Plan (kak u Foodforecast) — kogda i skolko dopekati.

In [ ]:
# === KUMULYATIVNYJ PROFIL PO TOVARAM ===
# Dlya kazhdogo tovara: kakaya dolya dnevnyh prodazh sdelana k kazhdomu chasu

top12_food = df[mask_food].groupby('Номенклатура')['Кол-во'].sum().nlargest(12).index.tolist()

# Dlya kazhdogo tovara x den x chas: qty
prod_daily_hour = (df[df['Номенклатура'].isin(top12_food)]
                   .groupby(['Номенклатура', 'Дата продажи', 'hour'])['Кол-во']
                   .sum().reset_index())

# Dnevnoj total po tovaru
prod_daily_total = (df[df['Номенклатура'].isin(top12_food)]
                    .groupby(['Номенклатура', 'Дата продажи'])['Кол-во']
                    .sum().rename('day_total').reset_index())

prod_daily_hour = prod_daily_hour.merge(prod_daily_total, on=['Номенклатура', 'Дата продажи'])
prod_daily_hour['share'] = prod_daily_hour['Кол-во'] / prod_daily_hour['day_total']

# Srednij kumulyativnyj profil po tovaru
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.ravel()

for i, prod in enumerate(top12_food):
    ax = axes[i]
    prod_data = prod_daily_hour[prod_daily_hour['Номенклатура'] == prod]
    avg_share = prod_data.groupby('hour')['share'].mean()
    cum = avg_share.cumsum()
    
    # Takzhe budni / vyhodnye
    prod_data_merged = prod_data.copy()
    prod_data_merged['dow'] = pd.to_datetime(prod_data_merged['Дата продажи']).dt.dayofweek
    
    cum_wd = prod_data_merged[prod_data_merged['dow'] < 5].groupby('hour')['share'].mean().cumsum()
    cum_we = prod_data_merged[prod_data_merged['dow'] >= 5].groupby('hour')['share'].mean().cumsum()
    
    ax.plot(cum.index, cum.values * 100, 'ko-', ms=4, lw=2, label='Vse')
    if len(cum_wd) > 0:
        ax.plot(cum_wd.index, cum_wd.values * 100, 's--', color='#2196F3', ms=3, label='Bud')
    if len(cum_we) > 0:
        ax.plot(cum_we.index, cum_we.values * 100, '^--', color='#FF5722', ms=3, label='Vyh')
    
    ax.axhline(50, color='gray', ls=':', alpha=0.4)
    ax.axhline(80, color='gray', ls=':', alpha=0.4)
    ax.set_title(prod[:22], fontsize=9)
    ax.set_xlim(5.5, 19.5)
    ax.set_ylim(0, 105)
    ax.set_xticks([6, 8, 10, 12, 14, 16, 18])
    if i >= 8:
        ax.set_xlabel('Chas')
    if i % 4 == 0:
        ax.set_ylabel('Cum %')
    ax.legend(fontsize=6, loc='lower right')
    ax.grid(True, alpha=0.2)

plt.suptitle('Kumulyativnyj profil prodazh po tovaram (baza dlya Production Plan)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Tablitsa: k kakomu chasu prodano 50% i 80%
print(f"\n{'Tovar':<35} {'50% k':>6} {'80% k':>6} {'Tip':>10}")
print("-" * 60)
for prod in top12_food:
    prod_data = prod_daily_hour[prod_daily_hour['Номенклатура'] == prod]
    avg_share = prod_data.groupby('hour')['share'].mean()
    cum = avg_share.cumsum()
    
    h50 = cum[cum >= 0.50].index[0] if (cum >= 0.50).any() else '?'
    h80 = cum[cum >= 0.80].index[0] if (cum >= 0.80).any() else '?'
    
    # Klassifikatsiya
    if isinstance(h50, int) and h50 <= 10:
        tip = 'UTRENNIJ'
    elif isinstance(h50, int) and h50 <= 12:
        tip = 'OBEDENNIJ'
    else:
        tip = 'RAVNOMERN'
    
    print(f"{prod:<35} {str(h50)+':00':>6} {str(h80)+':00':>6} {tip:>10}")

## 15. Korzinnyj analiz: chto pokupayut vmeste

Kakie tovary chashhe vsego okazyvayutsya v odnom cheke?
Pomogaet ponyat klasterizatsiyu tovarov (utrennij nabor, obedennij nabor i td).

In [ ]:
# === KORZINNYJ ANALIZ ===
from itertools import combinations
from collections import Counter

# Sozdaem korziny: gruppiruem po vremeni cheka (odin timestamp = odin chek)
baskets = df.groupby('Дата время чека')['Номенклатура'].apply(list).values

# TOP-20 food products dlya analiza
top20_food = df[mask_food].groupby('Номенклатура')['Кол-во'].sum().nlargest(20).index.tolist()
top20_set = set(top20_food)

# Schitaem pary
pair_counts = Counter()
single_counts = Counter()
n_baskets_multi = 0

for basket in baskets:
    # Filtr tolko TOP-20 food
    items = list(set([x for x in basket if x in top20_set]))
    for item in items:
        single_counts[item] += 1
    if len(items) >= 2:
        n_baskets_multi += 1
        for pair in combinations(sorted(items), 2):
            pair_counts[pair] += 1

print(f"Vsego korzin: {len(baskets):,}")
print(f"Korzin s 2+ TOP-20 food tovarami: {n_baskets_multi:,}")
print(f"\nTOP-20 par tovarov (po chastote sovmestnoy pokupki):")
print(f"\n{'Para':<55} {'Count':>6} {'Lift':>6}")
print("-" * 70)

total_baskets = len(baskets)
for (a, b), count in pair_counts.most_common(20):
    # Lift = P(A&B) / (P(A) * P(B))
    p_ab = count / total_baskets
    p_a = single_counts[a] / total_baskets
    p_b = single_counts[b] / total_baskets
    lift = p_ab / (p_a * p_b) if p_a * p_b > 0 else 0
    print(f"  {a[:25]:<25} + {b[:25]:<25} {count:>5}  {lift:>5.1f}")

# === Vizualizatsiya: matritsa sovmestnykh pokupok ===
top15_food = df[mask_food].groupby('Номенклатура')['Кол-во'].sum().nlargest(15).index.tolist()

co_matrix = pd.DataFrame(0, index=top15_food, columns=top15_food)
for (a, b), count in pair_counts.items():
    if a in top15_food and b in top15_food:
        co_matrix.loc[a, b] = count
        co_matrix.loc[b, a] = count

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(co_matrix.values, cmap='YlOrRd')
ax.set_xticks(range(len(top15_food)))
ax.set_yticks(range(len(top15_food)))
ax.set_xticklabels([n[:18] for n in top15_food], rotation=45, ha='right', fontsize=8)
ax.set_yticklabels([n[:18] for n in top15_food], fontsize=8)
ax.set_title('Matritsa sovmestnykh pokupok (TOP-15 food)')

for i in range(len(top15_food)):
    for j in range(len(top15_food)):
        val = co_matrix.values[i, j]
        if val > 0:
            color = 'white' if val > co_matrix.values.max() * 0.5 else 'black'
            ax.text(j, i, f'{val}', ha='center', va='center', fontsize=7, color=color)

plt.colorbar(im, label='Sovmestnye pokupki')
plt.tight_layout()
plt.show()

# === Klasterizatsiya: utrennij vs obedennij nabor ===
print(f"\n=== KLASTERIZATSIYA PO VREMENI POKUPKI ===")
print(f"\nSrednij chas pokupki dlya TOP-20 food:")
prod_avg_hour = (df[df['Номенклатура'].isin(top20_food)]
                 .groupby('Номенклатура')['hour'].mean()
                 .sort_values())

for name, h in prod_avg_hour.items():
    if h <= 10:
        cluster = 'UTRENNIJ'
    elif h <= 13:
        cluster = 'OBEDENNIJ'
    else:
        cluster = 'DNEVNOJ'
    bar = '#' * int((h - 6) * 3)
    print(f"  {name:<35} {h:>5.1f}  {bar}  [{cluster}]")

## 16. Otsenka upushchennogo sprosa cherez individualnyj profil

**Metod:** dlya kazhdogo tovara stroim kumulyativnyj profil sprosa iz "polnyh" dnej 
(poslednaya prodazha >= 17:00 — tovar ne zakonchilsya). 
Etot profil pokazyvaet, kakaya dolya dnevnyh prodazh sdelana k kazhdomu chasu.

Zatem dlya dnej, kogda tovar zakonchilsya rano (chas stopa X):
```
Ocenochnyj polnyj spros = faktich_prodazhi_do_X / kumulyativnaya_dolya(X)
Upushcheno = ocenochnyj_polnyj - faktich_prodazhi_do_X
```

Primer: Kish griby kuritsa, profil: k 12:00 prodano 55%. V den stopa prodali 6 sht, stop v 12:00.
→ Ocenochnyj polnyj = 6 / 0.55 = 10.9. Upushcheno = 4.9 sht.

Preimushchestvo pered "ploskoy" otsenkoy: uchityvaem KOGDA IMENNO tovar zakonchilsya, 
a ne prosto srednij hvost. Stop v 10:00 dast bolshuyu otsenku, chem stop v 15:00.

In [ ]:
# === 16. UPUSHCHENNYJ SPROS CHEREZ INDIVIDUALNYJ PROFIL ===

FULL_DAY_THRESHOLD = 17   # "polnyj den" = poslednaya prodazha >= 17:00
CLOSING_HOUR = 19

# =====================================================
# SHAG 1: Stroim kumulyativnyj profil dlya kazhdogo tovara
#         tolko iz "polnyh" dnej (tovar ne zakonchilsya)
#
# VAZHNO: cumsum schitaem VNUTRI kazhdogo dnya, potom
#          usrednyaem uzhe kumulyativnye znacheniya.
#          Eto garantiruet profil 0% -> 100%.
# =====================================================

# Dlya kazhdogo tovara x den x chas: qty
prod_hour_day = (df[df['Номенклатура'].isin(top30_food)]
                 .groupby(['Номенклатура', 'Дата продажи', 'hour'])['Кол-во']
                 .sum().reset_index())

# Dobavlyaem poslednij chas prodazhi v tot den
prod_hour_day = prod_hour_day.merge(
    last_sale[['Дата продажи', 'Номенклатура', 'last_hour']],
    on=['Дата продажи', 'Номенклатура'], how='left'
)

# Polnye dni
prod_hour_day['is_full_day'] = prod_hour_day['last_hour'] >= FULL_DAY_THRESHOLD

# Rabotaem tolko s polnymi dnyami dlya profilya
prod_full = prod_hour_day[prod_hour_day['is_full_day']].copy()

# Stroim profil: cumsum PER DAY, potom average
profiles = {}
for prod in top30_food:
    pdata = prod_full[prod_full['Номенклатура'] == prod]
    unique_days = pdata['Дата продажи'].unique()
    
    if len(unique_days) < 5:  # minimum 5 polnyh dnej
        continue
    
    cum_per_day = []
    for day in unique_days:
        day_data = pdata[pdata['Дата продажи'] == day]
        day_total = day_data['Кол-во'].sum()
        if day_total == 0:
            continue
        hourly = day_data.groupby('hour')['Кол-во'].sum().sort_index()
        # Kumulyativnyj profil VNUTRI dnya: 0 -> 1.0
        cum_day = hourly.cumsum() / day_total
        cum_per_day.append(cum_day)
    
    if len(cum_per_day) < 5:
        continue
    
    # Usrednyaem kumulyativnye profili po dnyam
    all_cum = pd.DataFrame(cum_per_day).sort_index(axis=1)
    avg_cum = all_cum.mean()  # teper eto srednyaya kumulyativnaya dolya
    # ffill chtoby zapolnit propushchennye chasy (esli v kakie-to chasy ne bylo prodazh)
    avg_cum = avg_cum.ffill()
    profiles[prod] = avg_cum

print(f"Postroeny profili dlya {len(profiles)} tovarov (iz {len(top30_food)} TOP-30)")
print(f"(minimum 5 polnyh dnej dlya nadezhnosti profilya)\n")

# Proverka: profil dolzhen zakanchivatcya okolo 100%
print("Proverka profilya (poslednyaya tochka dolzhna byt ~100%):")
for prod in list(profiles.keys())[:5]:
    cum = profiles[prod]
    print(f"  {prod[:28]:<28} max={cum.max()*100:.1f}%, last_hour={cum.index[-1]}")

# Pokaz profilya dlya 6 tovarov s naibolshim rannim stopom
show_prods = [p for p in stop_stats.head(12).index if p in profiles][:6]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()
for i, prod in enumerate(show_prods):
    ax = axes[i]
    cum = profiles[prod]
    ax.plot(cum.index, cum.values * 100, 'ko-', ms=5, lw=2)
    ax.fill_between(cum.index, 0, cum.values * 100, alpha=0.15, color='blue')
    
    ax.axhline(50, color='gray', ls=':', alpha=0.4)
    ax.axhline(100, color='gray', ls=':', alpha=0.4)
    ax.set_title(prod[:25], fontsize=10)
    ax.set_xlim(5.5, 19.5)
    ax.set_ylim(0, 110)
    ax.set_xticks(range(6, 20, 2))
    ax.set_xticklabels([f'{h}:00' for h in range(6, 20, 2)], fontsize=8)
    ax.set_ylabel('Kum. dolya (%)')
    ax.set_xlabel('Chas')
    ax.grid(True, alpha=0.2)
    
    # Annotiruem 50%
    h50_candidates = cum[cum >= 0.50]
    if len(h50_candidates) > 0:
        h50 = h50_candidates.index[0]
        ax.annotate(f'50% k {h50}:00', xy=(h50, 50), fontsize=8,
                   xytext=(h50+1, 60), arrowprops=dict(arrowstyle='->', color='red'))

plt.suptitle('Kumulyativnyj profil sprosa (iz polnyh dnej) — baza dlya otsenki', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Tablitsa profilya
print(f"\n{'Tovar':<30} {'PolnDn':>6}  Profil: k 8h  10h  12h  14h  16h  18h")
print("-" * 85)
for prod in top30_food:
    if prod not in profiles:
        continue
    cum = profiles[prod]
    n_full = len([d for d in prod_full[prod_full['Номенклатура'] == prod]['Дата продажи'].unique()])
    vals = []
    for h in [8, 10, 12, 14, 16, 18]:
        # Berem blizhaijshij chas <= h
        available = cum.index[cum.index <= h]
        if len(available) > 0:
            val = cum.loc[available[-1]] * 100
        else:
            val = 0
        vals.append(f"{val:>4.0f}%")
    print(f"{prod:<30} {n_full:>6}  {'  '.join(vals)}")

In [ ]:
# =====================================================
# SHAG 2: Primenyaem profil k dnyam s rannim stopom
#         Dlya kazhdogo dnya: estimated_full = actual_sold / cum_profile(stop_hour)
#         lost = estimated_full - actual_sold
#
# VAZHNO: estimated >= actual (esli profil dast menshe — 
#         znachit v tot den spros byl vyshe srednego,
#         i my beryom minimum = actual, lost = 0)
# =====================================================

day_results = []

for prod in top30_food:
    if prod not in profiles:
        continue
    
    cum = profiles[prod]
    avg_price = df[df['Номенклатура'] == prod]['Цена'].mean()
    
    prod_days = prod_hour_day[prod_hour_day['Номенклатура'] == prod]
    
    for day, day_data in prod_days.groupby('Дата продажи'):
        stop_hour = day_data['last_hour'].iloc[0]
        actual_sold = day_data['Кол-во'].sum()
        is_full = stop_hour >= FULL_DAY_THRESHOLD
        
        if is_full:
            day_results.append({
                'product': prod, 'date': day, 'stop_hour': stop_hour,
                'actual_sold': actual_sold, 'estimated_full': actual_sold,
                'lost_qty': 0, 'lost_revenue': 0, 'is_full_day': True,
                'avg_price': avg_price, 'cum_at_stop': 1.0
            })
            continue
        
        # Blizhaijshij chas v profile <= stop_hour
        available_hours = cum.index[cum.index <= stop_hour]
        if len(available_hours) == 0:
            continue
        
        cum_at_stop = cum.loc[available_hours[-1]]
        
        # Zaschita: profil < 5% — ne ekstrapoliruem
        if cum_at_stop < 0.05:
            continue
        
        estimated_full = actual_sold / cum_at_stop
        
        # Zaschita: cap x5
        estimated_full = min(estimated_full, actual_sold * 5)
        
        # estimated ne mozhet byt menshe actual
        estimated_full = max(estimated_full, actual_sold)
        
        lost_qty = estimated_full - actual_sold
        lost_revenue = lost_qty * avg_price
        
        day_results.append({
            'product': prod, 'date': day, 'stop_hour': stop_hour,
            'actual_sold': actual_sold, 'estimated_full': estimated_full,
            'lost_qty': lost_qty, 'lost_revenue': lost_revenue,
            'is_full_day': False, 'avg_price': avg_price,
            'cum_at_stop': cum_at_stop
        })

day_df = pd.DataFrame(day_results)
early_df = day_df[~day_df['is_full_day']].copy()

print(f"Vsego tovar-dnej: {len(day_df):,}")
print(f"Iz nih s rannim stopom: {len(early_df):,}")
print(f"Srednyaya otsenka upushchennogo (na 1 den stopa): {early_df['lost_qty'].mean():.1f} sht")
print(f"Srednij multiplikator: x{(early_df['estimated_full']/early_df['actual_sold']).mean():.2f}")
print(f"Srednij cum_at_stop: {early_df['cum_at_stop'].mean():.2f} ({early_df['cum_at_stop'].mean()*100:.0f}%)")
print()

# =====================================================
# SHAG 3: Agregiruem po tovaram
# =====================================================

product_lost = early_df.groupby('product').agg(
    n_early_days=('date', 'count'),
    avg_stop_hour=('stop_hour', 'mean'),
    avg_actual=('actual_sold', 'mean'),
    avg_estimated=('estimated_full', 'mean'),
    avg_lost_qty=('lost_qty', 'mean'),
    total_lost_qty=('lost_qty', 'sum'),
    total_lost_revenue=('lost_revenue', 'sum'),
    avg_price=('avg_price', 'first'),
    avg_cum_at_stop=('cum_at_stop', 'mean')
).sort_values('total_lost_revenue', ascending=False)

# Dobavlyaem info iz stop_stats
for prod in product_lost.index:
    if prod in stop_stats.index:
        product_lost.loc[prod, 'total_days'] = stop_stats.loc[prod, 'total_days']
        product_lost.loc[prod, 'early_pct'] = stop_stats.loc[prod, 'early_stop_pct']

product_lost['avg_multiplier'] = product_lost['avg_estimated'] / product_lost['avg_actual']

# =====================================================
# TABLITSA REZULTATOV
# =====================================================
print("=" * 120)
print("UPUSHCHENNYJ SPROS: otsenka cherez individualnyj profil")
print("=" * 120)
print(f"\n{'Tovar':<28} {'StopDn':>6} {'SrStop':>6} {'SrProf':>6} {'SrFakt':>6} {'SrOcen':>6} "
      f"{'x':>4} {'SrUpush':>7} {'VsegoSht':>8} {'VsegoRub':>9}")
print("-" * 115)

for prod, r in product_lost.iterrows():
    early_pct = r['early_pct'] if 'early_pct' in r and pd.notna(r['early_pct']) else 0
    marker = ''
    if r['total_lost_revenue'] > 3000 and early_pct > 20:
        marker = ' <<<'
    print(f"{prod:<28} {r['n_early_days']:>6.0f} {r['avg_stop_hour']:>5.1f}h "
          f"{r['avg_cum_at_stop']*100:>4.0f}% {r['avg_actual']:>6.1f} {r['avg_estimated']:>6.1f} "
          f"{r['avg_multiplier']:>3.1f}x {r['avg_lost_qty']:>7.1f} "
          f"{r['total_lost_qty']:>8.0f} {r['total_lost_revenue']:>9,.0f}{marker}")

# =====================================================
# VIZUALIZATSIYA 1: Primer profilya + den stopa
# =====================================================
example_prod = [p for p in stop_stats.head(5).index if p in profiles][0]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Levyj: profil + primer
ax = axes[0]
cum = profiles[example_prod]
ax.plot(cum.index, cum.values * 100, 'ko-', ms=6, lw=2, label='Kumulyativnyj profil (polnye dni)')
ax.fill_between(cum.index, 0, cum.values * 100, alpha=0.1, color='blue')

# Berem den s rannim stopom (chem ranshe, tem naglyandnee)
ex_early_data = early_df[early_df['product'] == example_prod].sort_values('stop_hour')
ex_early = ex_early_data.iloc[min(2, len(ex_early_data)-1)]  # ne samyj rannij, no rannij
stop_h = int(ex_early['stop_hour'])
actual = ex_early['actual_sold']
estimated = ex_early['estimated_full']
cum_at_s = ex_early['cum_at_stop']

ax.axvline(stop_h, color='red', ls='--', lw=2, label=f'Stop v {stop_h}:00')
ax.axhline(cum_at_s * 100, color='red', ls=':', alpha=0.5)

# Zashtrihovannaya zona = upushchennyj spros
ax.fill_between([stop_h, cum.index[-1]], [cum_at_s*100, cum_at_s*100], [100, 100],
                alpha=0.15, color='red', label='Upushchennyj spros')

ax.annotate(f'Profil k {stop_h}:00 = {cum_at_s*100:.0f}%\n'
            f'Fakt: {actual:.0f} sht\n'
            f'Ocenka polnogo: {actual:.0f}/{cum_at_s:.2f} = {estimated:.0f} sht\n'
            f'Upushcheno: {estimated-actual:.0f} sht',
            xy=(stop_h, cum_at_s * 100),
            xytext=(stop_h + 1.5, max(cum_at_s * 100 - 25, 10)),
            fontsize=9, bbox=dict(boxstyle='round', facecolor='lightyellow'),
            arrowprops=dict(arrowstyle='->', color='red'))

ax.set_title(f'Primer: {example_prod[:25]}', fontsize=11)
ax.set_xlabel('Chas')
ax.set_ylabel('Kumulyativnaya dolya (%)')
ax.set_xlim(5.5, 19.5)
ax.set_ylim(0, 110)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

# Pravyj: multiplikator po tovaram
ax = axes[1]
mult_data = product_lost.sort_values('avg_multiplier', ascending=True)
colors_m = ['#FF5722' if m > 1.5 else '#FFC107' if m > 1.2 else '#4CAF50' 
            for m in mult_data['avg_multiplier']]
ax.barh(range(len(mult_data)), mult_data['avg_multiplier'], color=colors_m, alpha=0.8)
ax.set_yticks(range(len(mult_data)))
ax.set_yticklabels([n[:22] for n in mult_data.index], fontsize=7)
ax.axvline(1.0, color='black', ls='-', lw=1)
ax.axvline(1.5, color='red', ls=':', alpha=0.5, label='x1.5')
ax.set_xlabel('Srednij multiplikator (ocenka/fakt)')
ax.set_title('Vo skolko raz realnyj spros vyshe prodazh\n(v dni rannego stopa)')
ax.legend()

plt.tight_layout()
plt.show()

# =====================================================
# VIZUALIZATSIYA 2: Upushchennaya vyruchka + scatter
# =====================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

ax = axes[0]
x = product_lost['early_pct'].fillna(0)
y = product_lost['total_lost_revenue']
sizes = product_lost['avg_multiplier'] * 100

colors_s = ['#FF5722' if rev > 3000 and pct > 20
            else '#FFC107' if rev > 1500
            else '#4CAF50' 
            for rev, pct in zip(y, x)]

ax.scatter(x, y, s=sizes, c=colors_s, alpha=0.7, edgecolors='black', linewidths=0.5)

for prod, r in product_lost.iterrows():
    ep = r['early_pct'] if pd.notna(r.get('early_pct')) else 0
    if r['total_lost_revenue'] > 2000 or ep > 30:
        ax.annotate(prod[:15], (ep, r['total_lost_revenue']),
                   fontsize=7, ha='left', va='bottom')

ax.set_xlabel('% dnej s rannim stopom')
ax.set_ylabel('Upushchennaya vyruchka za 3 mes. (rub)')
ax.set_title('Upushchennaya vyruchka vs chastota stopa\n(razmer = multiplikator sprosa)')
ax.grid(True, alpha=0.2)

# Bar: TOP-15
ax = axes[1]
top15 = product_lost.head(15)
colors_bar = ['#FF5722' if pd.notna(r.get('early_pct')) and r['early_pct'] > 20 
              else '#FFC107' for _, r in top15.iterrows()]
ax.barh(range(len(top15)), top15['total_lost_revenue'], color=colors_bar, alpha=0.8)
ax.set_yticks(range(len(top15)))
ax.set_yticklabels([p[:22] for p in top15.index], fontsize=8)
ax.set_xlabel('Upushchennaya vyruchka (rub) za 3 mes.')
ax.set_title('TOP-15 po upushchennoj vyruchke\n(krasnyj = chastye stopy >20%)')

for i, (prod, r) in enumerate(top15.iterrows()):
    ax.text(r['total_lost_revenue'] + 50, i, 
            f"{r['total_lost_revenue']:,.0f} (x{r['avg_multiplier']:.1f})",
            va='center', fontsize=7)

plt.tight_layout()
plt.show()

# =====================================================
# VIZUALIZATSIYA 3: Raspredelenie po chasam stopa
# =====================================================
fig, ax = plt.subplots(figsize=(14, 5))

hour_lost = early_df.groupby('stop_hour').agg(
    n_cases=('date', 'count'),
    avg_lost=('lost_qty', 'mean'),
    total_lost_rev=('lost_revenue', 'sum')
)

ax2 = ax.twinx()
ax.bar(hour_lost.index, hour_lost['total_lost_rev'], color='#FF5722', alpha=0.7, label='Vsego upushcheno (rub)')
ax2.plot(hour_lost.index, hour_lost['avg_lost'], 'ko-', ms=6, lw=2, label='Sr. upushcheno (sht)')

ax.set_xlabel('Chas poslednej prodazhi (stop)')
ax.set_ylabel('Obshaya upushchennaya vyruchka (rub)')
ax2.set_ylabel('Srednee upushchennoe kol-vo (sht)')
ax.set_title('Upushchennyj spros v zavisimosti ot chasa stopa\n(chem ranshe stop — tem bolshe teryaem)')
ax.set_xticks(range(6, 19))
ax.set_xticklabels([f'{h}:00' for h in range(6, 19)])
ax.legend(loc='upper right')
ax2.legend(loc='upper left')

plt.tight_layout()
plt.show()

print(f"\nUpushchennyj spros po chasu stopa:")
print(f"{'Chas stopa':>10} {'Sluchaev':>8} {'Sr.upush sht':>12} {'Vsego rub':>10}")
print("-" * 45)
for h, r in hour_lost.iterrows():
    print(f"{h:>9}h {r['n_cases']:>8.0f} {r['avg_lost']:>12.1f} {r['total_lost_rev']:>10,.0f}")

# =====================================================
# ITOG
# =====================================================
total_lost_rev = product_lost['total_lost_revenue'].sum()
top_lost = product_lost[product_lost['early_pct'].fillna(0) > 20]
top_lost_rev = top_lost['total_lost_revenue'].sum()

print(f"\n{'='*70}")
print(f"ITOG: Upushchennyj spros za 3 mesyatsa (metod kumulyativnogo profilya)")
print(f"{'='*70}")
print(f"  Vsego po TOP-30 tovaram:           {total_lost_rev:>12,.0f} rub")
print(f"  V mesyats:                          {total_lost_rev/3:>12,.0f} rub")
print(f"  Tovary s chastymi stopami (>20%):   {top_lost_rev:>12,.0f} rub")
print(f"  V mesyats:                          {top_lost_rev/3:>12,.0f} rub")

print(f"\n  TOP-5 po upushchennoj vyruchke:")
for prod, r in product_lost.head(5).iterrows():
    print(f"    {prod:<28} {r['total_lost_revenue']:>9,.0f} rub "
          f"(x{r['avg_multiplier']:.1f}, {r['n_early_days']:.0f} dnej stopa, "
          f"sr.profil={r['avg_cum_at_stop']*100:.0f}%)")

print(f"\n  Sravnenie s prostyni metodom (Section 16 v1):")
print(f"    Prostoj (hvost iz polnyh dnej):  ~98,000 rub/3 mes")
print(f"    Profilnyj (tekushchij):          {total_lost_rev:>9,.0f} rub/3 mes")
print(f"    Raznitsa ob'yasnyaetsya tem, chto bolshinstvo stopov")
print(f"    proiskhodyat pozdno (sr. {early_df['stop_hour'].mean():.0f}h), kogda po profilyu")
print(f"    uzhe prodano {early_df['cum_at_stop'].mean()*100:.0f}% dnevnogo ob'yoma.")

print(f"\n  Metod: individualnyj kumulyativnyj profil sprosa")
print(f"  Baza: 'polnye' dni (poslednaya prodazha >= {FULL_DAY_THRESHOLD}:00)")
print(f"  Formula: estimated = max(actual, actual / cum_profile(stop_hour))")
print(f"  Zaschita: cum_profile >= 5%, multiplikator <= x5")